## SEINE Video Generation model

In [1]:
import os
import sys
import math
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'SEINE'))

import utils
from diffusion import create_diffusion

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import argparse
import torchvision

from einops import rearrange
from models import get_models
from torchvision.utils import save_image
from diffusers.models import AutoencoderKL
from models.clip import TextEmbedder
from omegaconf import OmegaConf
from PIL import Image
import numpy as np
from torchvision import transforms
from SEINE import video_transforms
# from dataset import video_transforms
from utils import mask_generation_before
from natsort import natsorted
from diffusers.utils.import_utils import is_xformers_available
import pdb

class SeineModel:
    def __init__(self, args):
        print('Initializing SEINE model...')

        if args.seed:
            torch.manual_seed(args.seed)
        torch.set_grad_enabled(False)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if args.ckpt is None:
            raise ValueError("Please specify a checkpoint path using --ckpt <path>")

        # load model
        self.latent_h = args.image_size[0] // 8
        self.latent_w = args.image_size[1] // 8
        self.image_h = args.image_size[0]
        self.image_w = args.image_size[1]
        self.model = get_models(args).to(self.device)

        if args.enable_xformers_memory_efficient_attention:
            if is_xformers_available():
                self.model.enable_xformers_memory_efficient_attention()
            else:
                raise ValueError("xformers is not available. Make sure it is installed correctly")

        ckpt_path = args.ckpt 
        state_dict = torch.load(ckpt_path, map_location=lambda storage, loc: storage)['ema']
        self.model.load_state_dict(state_dict)

        self.model.eval()
        pretrained_model_path = args.pretrained_model_path
        self.diffusion = create_diffusion(str(args.num_sampling_steps))
        self.vae = AutoencoderKL.from_pretrained(pretrained_model_path, subfolder="vae").to(self.device)
        self.text_encoder = TextEmbedder(pretrained_model_path).to(self.device)
        if args.use_fp16:
            # print('Warning: using half percision for inferencing!')
            self.vae.to(dtype=torch.float16)
            self.model.to(dtype=torch.float16)
            self.text_encoder.to(dtype=torch.float16)

        self.mask_type = args.mask_type
        self.num_frames = args.num_frames
        self.use_fp16 = args.use_fp16
        self.do_classifier_free_guidance = args.do_classifier_free_guidance
        self.sample_method = args.sample_method
        self.cfg_scale = args.cfg_scale
        self.use_mask = args.use_mask

        print('Initialization complete!')

    def get_input(self, input_path):
        transform_video = transforms.Compose([
                            video_transforms.ToTensorVideo(), # TCHW
                            video_transforms.ResizeVideo((self.image_h, self.image_w)),
                            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
                        ])
        if input_path is not None:
            print(f'Loading video from {input_path}...')
            if os.path.isdir(input_path):
                file_list = os.listdir(input_path)
                video_frames = []
                if self.mask_type.startswith('onelast'):
                    num = int(self.mask_type.split('onelast')[-1])
                    # get first and last frame
                    first_frame_path = os.path.join(input_path, natsorted(file_list)[0])
                    last_frame_path = os.path.join(input_path, natsorted(file_list)[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(first_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    last_frame = torch.as_tensor(np.array(Image.open(last_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    # add zeros to frames
                    num_zeros = self.num_frames-2*num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    for i in range(num):
                        video_frames.append(last_frame)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                else:
                    for file in file_list:
                        if file.endswith('jpg') or file.endswith('png'):
                            image = torch.as_tensor(np.array(Image.open(file), dtype=np.uint8, copy=True)).unsqueeze(0)
                            video_frames.append(image)
                        else:
                            continue
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                return video_frames, n
            elif os.path.isfile(input_path):
                _, full_file_name = os.path.split(input_path)
                file_name, extension = os.path.splitext(full_file_name)
                if extension == '.jpg' or extension == '.png':
                    print("Loading the input image...")
                    video_frames = []
                    num = int(self.mask_type.split('first')[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(input_path).convert('RGB'), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    num_zeros = self.num_frames-num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                    return video_frames, n
                else:
                    raise TypeError(f'{extension} is not supported !!')
            else:
                raise ValueError('Please check your path input!!')
        else:
            raise ValueError('Need to give a video or some images')

    def auto_inpainting(self, video_input, masked_video, mask, prompt, negative_prompt):
        b,f,c,h,w = video_input.shape

        # prepare inputs
        if self.use_fp16:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, dtype=torch.float16, device=self.device) # b,c,f,h,w
            masked_video = masked_video.to(dtype=torch.float16)
            mask = mask.to(dtype=torch.float16)
        else:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, device=self.device) # b,c,f,h,w

        masked_video = rearrange(masked_video, 'b f c h w -> (b f) c h w').contiguous()
        masked_video = self.vae.encode(masked_video).latent_dist.sample().mul_(0.18215)
        masked_video = rearrange(masked_video, '(b f) c h w -> b c f h w', b=b).contiguous()
        mask = torch.nn.functional.interpolate(mask[:,:,0,:], size=(self.latent_h, self.latent_w)).unsqueeze(1)
    
        # classifier_free_guidance
        if self.do_classifier_free_guidance:
            masked_video = torch.cat([masked_video] * 2)
            mask = torch.cat([mask] * 2)
            z = torch.cat([z] * 2)
            prompt_all = [prompt] + [negative_prompt]

        else:
            masked_video = masked_video
            mask = mask
            z = z
            prompt_all = [prompt]

        text_prompt = self.text_encoder(text_prompts=prompt_all, train=False)
        model_kwargs = dict(encoder_hidden_states=text_prompt, 
                                class_labels=None, 
                                cfg_scale=self.cfg_scale,
                                use_fp16=self.use_fp16,) # tav unet

        # sample video
        if self.sample_method == 'ddim':
            samples = self.diffusion.ddim_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        elif self.sample_method == 'ddpm':
            samples = self.diffusion.p_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        samples, _ = samples.chunk(2, dim=0) # [1, 4, 16, 32, 32]
        if self.use_fp16:
            samples = samples.to(dtype=torch.float16)

        video_clip = samples[0].permute(1, 0, 2, 3).contiguous() # [16, 4, 32, 32]
        video_clip = self.vae.decode(video_clip / 0.18215).sample # [16, 3, 256, 256]

        return video_clip

    def generate_video(self, args):
        prompt = args.text_prompt
        
        if prompt == []:
            prompt = args.input_path.split('/')[-1].split('.')[0].replace('_', ' ')
        else:
            prompt = prompt[0]
        prompt_base = prompt.replace(' ','_')

        if not os.path.exists(os.path.join(args.save_path)):
            os.makedirs(os.path.join(args.save_path))
        video_input, reserve_frames = self.get_input(args.input_path) # f,c,h,w
        video_input = video_input.to(self.device).unsqueeze(0)  # b,f,c,h,w
        mask = mask_generation_before(self.mask_type, video_input.shape, video_input.dtype, self.device) # b,f,c,h,w
        masked_video = video_input * (mask == 0)

        video_clip = self.auto_inpainting(video_input, masked_video, mask, prompt, args.negative_prompt)
        video_ = ((video_clip * 0.5 + 0.5) * 255).add_(0.5).clamp_(0, 255).to(dtype=torch.uint8).cpu().permute(0, 2, 3, 1)
        save_video_path = os.path.join(args.save_path, prompt_base+ '.mp4')
        torchvision.io.write_video(save_video_path, video_, fps=8)
        print(f'Video saved in {save_video_path}')


/home/brina/miniconda3/envs/seine/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load SEINE model with configs

In [2]:
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, default="./configs/seine.yaml")
args, unknown = parser.parse_known_args()
omega_conf = OmegaConf.load(args.config)
seine_model = SeineModel(omega_conf)

Initializing SEINE model...
Initialization complete!


## PromptPilot Agent

In [3]:
from openai import OpenAI
import yaml
import re
from argparse import Namespace
import base64

class PromptPilotAgent:
    def __init__(self, seine_model):
        self.llm = OpenAI(
                api_key="fy3jHNMV7OC7t7fQprQkFgp7NeSlRsMG",
                base_url="https://api.deepinfra.com/v1/openai",
            )
        self.temperature = 0.0
        self.max_iterations = 10
        self.seine_model = seine_model

        self.SYSTEM_PROMPT = """
                                You are a prompt refinement agent specialized in enhancing prompts for video generation models that take a single image as input.
                                Your goal is to improve the given prompt so that the generated video is visually coherent, smooth in motion, and logically consistent with the content and context of the image.
                                Carefully consider the visual elements and implied actions in the image, and rewrite the prompt to guide the model toward generating a realistic and temporally logical video sequence.
                             """
        
        self.USER_PROMPT = """
                              Given an image and the previous prompt, refine the prompt to result in a better video.  
                           """

        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]

    def set_first_text_prompt(self, yaml_path):
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)

        # Extract the input_path value to use as the new_prompt
        input_path = content.get("input_path", "No input_path found")
        prompt = input_path.split('/')[-1].split('.')[0].replace('_', ' ')
    
        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: [{prompt}]\n',
            content
        )

        with open(yaml_path, 'w') as f:
            f.write(new_content)

    def update_text_prompt(self, yaml_path, new_prompt):
        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: [{new_prompt}]\n',
            content
        )

        with open(yaml_path, 'w') as f:
            f.write(new_content)

    def query_llm(self):
        response = self.llm.chat.completions.create(
            # model="meta-llama/Llama-3.2-11B-Vision-Instruct",
            model="meta-llama/Llama-3.2-90B-Vision-Instruct",
            messages=self.messages,
            temperature=self.temperature
        )
        return response.choices[0].message.content

    def clean_response(self, response):
        match = re.search(r'"(.*?)"', response, re.DOTALL)
        if match:
            # extract quoted text it and remove newlines
            extracted_text = re.sub(r'[\n\r]', '', match.group(1))
        else:
            # clean newlines from the whole response
            extracted_text = re.sub(r'[\n\r]', '', response)

        # match text after "Refined prompt: "
        match = re.search(r'Refined prompt: (.*)', extracted_text)
        if match:
            extracted_text = match.group(1)

        return extracted_text
    
    def cleanup(self):
        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]
        torch.cuda.empty_cache()

    def run(self, yaml_path):
        print("Generating video...")

        # set text prompt for the first time
        self.set_first_text_prompt(yaml_path)

        # Read the file content
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)

        # get the string inside the text_prompt list
        prompt = content.get("text_prompt", [None])[0]
        input_path = content.get("input_path", None)

        # prepare llm image prompt
        with open(input_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode("utf-8")

        # prepare llm text prompt
        text = self.USER_PROMPT + \
                f"""
                    Previous prompt: {prompt}
                    Refined prompt:
                """

        # create llm use prompt
        self.messages.append(
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64, {encoded_image}"
                        }
                    },
                    {
                        "type": "text",
                        "text": text
                    }
                ]
            }
        )

        # get llm response
        response = self.query_llm()

        # clean up response
        extracted_text = self.clean_response(response)
        print(f"Refined prompt: {extracted_text}")

        # update yaml
        self.update_text_prompt(yaml_path, extracted_text)

        # prepare args for video generation
        with open(yaml_path, "r") as f:
            content = yaml.safe_load(f)
        args = Namespace(**content)
        self.seine_model.generate_video(args)

        # cleanup after generation
        self.cleanup()

## Create Prompt Agent

In [5]:
agent = PromptPilotAgent(seine_model)

directory = "configs/images"
yaml_paths = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.yaml')]

for yaml_path in yaml_paths:
    print(f"\nProcessing: {yaml_path}")
    agent.run(yaml_path)



Processing: configs/images/sea.yaml
Generating video...
Refined prompt: Capture the serene beauty of the ocean, with gentle waves rolling onto the shore and a vibrant sunset casting a warm glow on the water's surface.
Loading video from dataset/beauty_of_the_sea.png...
Loading the input image...


100%|██████████| 250/250 [02:48<00:00,  1.49it/s]


Video saved in ./results/Capture_the_serene_beauty_of_the_ocean.mp4

Processing: configs/images/pouring-essense.yaml
Generating video...
Refined prompt: A bottle of essence is being poured into a container, with the liquid flowing smoothly and steadily, creating a sense of movement and fluidity.
Loading video from dataset/essence_is_poured_from_a_bottle.png...
Loading the input image...


100%|██████████| 250/250 [02:52<00:00,  1.45it/s]


Video saved in ./results/A_bottle_of_essence_is_being_poured_into_a_container.mp4
